# Exploring the PDS Product Catalog

The `planetarypy.catalog` module provides access to a comprehensive catalog of data products across the entire NASA Planetary Data System (PDS) archive. It is built by ingesting instrument definitions from the [MillionConcepts pdr-tests](https://github.com/MillionConcepts/pdr-tests) repository, which has cataloged ~200 instruments across 60+ missions.

The catalog follows the **mission.instrument.product** dotted key convention, consistent with `planetarypy`'s existing PDS index system.

**Prerequisites:** Install with `pip install planetarypy[catalog]` to get the DuckDB dependency.

## Building the catalog

The first step is to build the catalog database. This clones the pdr-tests repository (sparse checkout, only the definitions folder) and parses all instrument definitions into a local DuckDB database.

This only needs to be done once. Subsequent calls will skip if the database already exists (use `force=True` to rebuild).

In [ ]:
# Optional: enable logging to see what's happening behind the scenes
from loguru import logger
logger.enable("planetarypy")

In [ ]:
from planetarypy.catalog import (
    build_catalog,
    list_missions,
    list_instruments,
    list_product_types,
    get_products,
    search,
    summary,
    ambiguous_mappings,
)

In [ ]:
stats = build_catalog()
stats

## Catalog overview

The `summary()` function gives a bird's-eye view of what the catalog contains, grouped by mission.

In [ ]:
summary()

## Navigating the hierarchy: missions, instruments, products

The catalog follows a three-level hierarchy: **mission** > **instrument** > **product type**.

You can drill down level by level, or use dotted keys to jump directly.

In [ ]:
# All missions in the catalog
list_missions()

In [ ]:
# What instruments does Cassini have?
list_instruments("cassini")

In [ ]:
# What product types exist for Cassini ISS?
# Both styles work: dotted key or separate arguments
list_product_types("cassini.iss")

In [ ]:
# Equivalent call with separate arguments
list_product_types("cassini", "iss")

### Exploring more missions

In [ ]:
# LRO instruments
list_instruments("lro")

In [ ]:
# Diviner products (note: Diviner is correctly mapped to LRO)
list_product_types("lro.diviner")

In [ ]:
# New Horizons instruments
list_instruments("new_horizons")

In [ ]:
# LORRI product types
list_product_types("new_horizons.lorri")

## Retrieving product details

The `get_products()` function returns a DataFrame with sample product entries for a given product type, including PDS archive URLs, file lists, and product IDs.

In [ ]:
# Get sample products for Cassini ISS Saturn EDRs
get_products("cassini.iss.edr_sat")

In [ ]:
# LRO Diviner EDR products
get_products("lro.diviner.edr")

In [ ]:
# Cassini RADAR BIDR products
get_products("cassini.radar.bidr")

## Searching across the catalog

The `search()` function lets you find products by keyword across missions, instruments, product types, and product IDs.

In [ ]:
search("hirise")

In [ ]:
# Search for spectrometer-related products
search("spectrometer")

In [ ]:
# Find anything related to Rosetta
search("rosetta")

## Downloading products

The catalog isn't just for discovery — you can download actual PDS data products directly. The `get_product()` function resolves a product to its remote URLs and downloads the files to a local directory.

### Resolution tiers

Product resolution works in two tiers:

1. **Catalog lookup** (Tier 1): For the ~1,948 sample products in the catalog database, the URL is known directly. This always works for products returned by `get_products()`.

2. **Index lookup** (Tier 2): For arbitrary product IDs, the system looks up the product in a PDS cumulative index (if one is registered for the instrument). This covers millions of additional products for instruments like CTX, HiRISE, Cassini ISS, Galileo SSI, and others.

In [ ]:
from planetarypy.catalog import get_product, get_product_url, list_product_files

### Inspecting product URLs without downloading

Use `get_product_url()` to see the remote base URL, or `list_product_files()` to see all files and their full URLs:

In [ ]:
# First, let's find a sample product to work with
products = get_products("cassini.iss.edr_sat")
sample_pid = products.iloc[0]["product_id"]
print(f"Sample product ID: {sample_pid}")

# Get the remote URL without downloading
url = get_product_url("cassini.iss.edr_sat", sample_pid)
print(f"Remote URL: {url}")

In [ ]:
# List all files and their full URLs for the product
list_product_files("cassini.iss.edr_sat", sample_pid)

### Downloading a product

`get_product()` downloads the product files and returns the local directory path. Files are cached — subsequent calls skip already-downloaded files.

In [ ]:
# Download a product — returns the local directory
# Uses dotted key: "mission.instrument.product_type"
# local_dir = get_product("cassini.iss.edr_sat", sample_pid)
# print(f"Downloaded to: {local_dir}")
# print(f"Files: {list(local_dir.iterdir())}")

# Options:
#   label_only=True  — download only the label file
#   force=True       — re-download even if files exist
#   files=["specific_file.IMG"]  — download specific files only

# Separate arguments also work:
# local_dir = get_product("cassini", sample_pid, instrument="iss", product_key="edr_sat")

### Index-backed resolution (Tier 2)

For instruments with PDS cumulative indexes, you can resolve *any* product ID — not just the sample products in the catalog. This works for CTX, HiRISE, Cassini ISS, Galileo SSI, LROC, Diviner, CRISM, and others.

The system automatically falls back to the index when the product isn't found in the catalog samples.

In [ ]:
from planetarypy.catalog._index_bridge import list_indexed_instruments, has_index

# Which instruments have index-backed resolution?
for mission, instrument, index_key in list_indexed_instruments():
    print(f"  {mission}.{instrument} → {index_key}")

In [ ]:
# For indexed instruments, you can resolve arbitrary product IDs:
# (This requires the PDS index to be downloaded — first call may take a moment)

# Example: get the URL for a specific CTX observation
# url = get_product_url("mro.ctx.edr", "B01_009942_1894_XI_09N202W")
# print(url)

# Example: download a specific HiRISE product
# local_dir = get_product("mro.hirise.edr", "PSP_001330_2530_RED0_0")

# Check if an instrument has index support
print(f"CTX has index: {has_index('mro', 'ctx')}")
print(f"LORRI has index: {has_index('new_horizons', 'lorri')}")

## Direct database access

For more advanced queries, you can get a DuckDB connection and write SQL directly. The catalog provides a convenient `catalog` view that joins instruments, product types, and products.

In [ ]:
from planetarypy.catalog import get_catalog

con = get_catalog()

In [ ]:
# Which missions have the most product types?
con.sql("""
    SELECT mission, COUNT(DISTINCT product_key) as n_product_types
    FROM catalog
    GROUP BY mission
    ORDER BY n_product_types DESC
    LIMIT 10
""").fetchdf()

In [ ]:
# Find all product types that have attached labels
con.sql("""
    SELECT mission, instrument, product_key
    FROM catalog
    WHERE label_type = 'A'
    GROUP BY mission, instrument, product_key
    ORDER BY mission, instrument
    LIMIT 20
""").fetchdf()

In [ ]:
# Explore the URL hosting patterns across the archive
con.sql("""
    SELECT
        CASE
            WHEN url_stem LIKE '%pdsimage2.wr.usgs.gov%' THEN 'USGS Imaging'
            WHEN url_stem LIKE '%pds-imaging.jpl.nasa.gov%' THEN 'JPL Imaging'
            WHEN url_stem LIKE '%planetarydata.jpl.nasa.gov%' THEN 'JPL Planetary Data'
            WHEN url_stem LIKE '%pds-atmospheres.nmsu.edu%' THEN 'Atmospheres (NMSU)'
            WHEN url_stem LIKE '%pds-geosciences.wustl.edu%' THEN 'Geosciences (WashU)'
            WHEN url_stem LIKE '%pds-rings.seti.org%' THEN 'Rings (SETI)'
            WHEN url_stem LIKE '%s3.%amazonaws.com%' THEN 'AWS S3 Mirror'
            WHEN url_stem LIKE '%hirise-pds.lpl.arizona.edu%' THEN 'HiRISE (Arizona)'
            WHEN url_stem LIKE '%pds-smallbodies%' THEN 'Small Bodies'
            WHEN url_stem LIKE '%archives.esac.esa.int%' THEN 'ESA PSA'
            ELSE 'Other'
        END as hosting_node,
        COUNT(*) as n_products
    FROM products
    WHERE url_stem IS NOT NULL AND url_stem != ''
    GROUP BY hosting_node
    ORDER BY n_products DESC
""").fetchdf()

In [ ]:
con.close()

## Mission mapping review

The catalog maps pdr-tests folder names (like `diviner`, `gal_ssi`, `vg_iss`) to proper mission/instrument tuples. Most mappings are automatic (split on underscore) or handled by a curated manual map.

Use `ambiguous_mappings()` to check if any folder names could not be confidently assigned.

In [ ]:
ambiguous_mappings()

## URL health notes

Not all product URLs in the catalog are still valid. During initial research, the following was found:

| Hosting Node | Status |
|---|---|
| USGS Imaging (`pdsimage2.wr.usgs.gov/Missions/*`) | **Broken** -- all 404 |
| JPL Imaging (`pds-imaging.jpl.nasa.gov`) | Migrated to `planetarydata.jpl.nasa.gov` (redirects work) |
| Atmospheres (NMSU) | Healthy |
| Geosciences (WashU) | Healthy |
| Rings (SETI) | Healthy (some path restructuring) |
| AWS S3 mirrors | Healthy |
| HiRISE (Arizona) | Healthy |

You can validate URLs in the catalog using the validation module:

In [ ]:
# Uncomment to run URL validation (takes a few minutes, checks sample URLs per product type)
# from planetarypy.catalog._validation import validate_urls
# from planetarypy.config import config
# counts = validate_urls(config.storage_root, sample_size=2)
# print(counts)

## Relationship to PDS indexes

The catalog module complements and integrates with the existing `planetarypy.pds` index system:

| | PDS Indexes (`planetarypy.pds`) | PDS Catalog (`planetarypy.catalog`) |
|---|---|---|
| **Scope** | ~30 curated index files | ~2000 product types across 200+ instruments |
| **Data** | Full observation-level metadata (DataFrames with 30-140 columns) | Product-level metadata (URLs, file lists, product IDs) |
| **Source** | PDS archive `.lbl` + `.tab` files | pdr-tests repository definitions |
| **Storage** | Parquet files | DuckDB database |
| **Use case** | Query specific observations | Discover and download data across the PDS |

**Integration**: The catalog's Tier 2 resolution automatically uses PDS indexes when available. For instruments like CTX, HiRISE, Cassini ISS, and Galileo SSI, this means you can resolve and download *any* product by ID — the catalog handles the URL construction, volume lookup, and archive-specific path conventions behind the scenes.

## Rebuilding the catalog

To update the catalog with the latest definitions from pdr-tests, use `force=True`:

```python
build_catalog(force=True)
```

This will re-clone the repository and rebuild the DuckDB database from scratch.